[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week10.ipynb)

# Week 10: Recurrent Networks (RNNs)
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Sequence data and recurrence; the RNN cell; backpropagation through time; vanishing and exploding gradients.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Build a plain RNN for sequence data.
- Understand recurrence and backpropagation through time.
- Observe the vanishing-gradient problem directly.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Build a plain RNN on a short sequence task and run it.

In [ ]:
# A plain RNN over a sequence
torch.manual_seed(0)
rnn = nn.RNN(input_size=1, hidden_size=16, batch_first=True)
x = torch.randn(8, 20, 1)                 # (batch, time, features)
out, h = rnn(x)
print("output sequence:", tuple(out.shape), "| final hidden:", tuple(h.shape))

## Demonstration 2
Plot gradient norms across time steps to expose vanishing gradients.

In [ ]:
# Gradient of the last output w.r.t. each input step: it vanishes for early steps
T = 40
rnn = nn.RNN(1, 8, batch_first=True)
x = torch.randn(1, T, 1, requires_grad=True)
out, _ = rnn(x)
out[:, -1].sum().backward()
g = x.grad.abs().squeeze().tolist()
plt.plot(range(T), g); plt.xlabel("time step"); plt.ylabel("|grad of last output|")
plt.title("Vanishing gradient"); plt.show()

## Demonstration 3
Demonstrate gradient clipping.

In [ ]:
# Gradient clipping caps the global gradient norm
rnn = nn.RNN(1, 8, batch_first=True); o = torch.optim.SGD(rnn.parameters(), 0.1)
x = torch.randn(4, 30, 1); y = torch.randn(4, 30, 8)
o.zero_grad(); out, _ = rnn(x); ((out - y) ** 2).mean().backward()
norm = torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)
print("grad norm before clip:", round(norm.item(), 3), "-> clipped to <= 1.0")

---
Student materials for this week: the lab handout (`labs/week10.html`) and the curated references (`references/week10.html`) in the course site.